# MAT benchmark, Tier 1 on Kaggle (2 x T4): tasks estrogen-alpha

All eight Tier-1 models (RF, SVM, GCN, MAT, three ablations, hybrid), 5 seeds x 5 folds x 2 splits, with the frozen configurations from results/tuning/best_configs.json (PRD section 3.2/3.3). Each GPU takes one split type; several worker processes share each GPU.

In [ ]:
import subprocess, sys, os, shutil, glob, time, zipfile
open('/kaggle/working/progress.txt', 'w').write(time.strftime('%H:%M:%S ') + 'notebook started\n')
try:
    print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], capture_output=True, text=True).stdout)
except Exception as e:
    print('nvidia-smi unavailable:', e)
import torch; N_GPU = torch.cuda.device_count(); print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), 'gpus', N_GPU)
print('cpus', os.cpu_count())
open('/kaggle/working/progress.txt', 'a').write(time.strftime('%H:%M:%S ') + f'gpus={N_GPU} cpus={os.cpu_count()}\n')

In [ ]:
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'rdkit', 'torch_geometric', 'threadpoolctl'], capture_output=True, text=True)
print(r.stdout[-800:], r.stderr[-800:])
open('/kaggle/working/progress.txt', 'a').write(time.strftime('%H:%M:%S ') + f'pip exit {r.returncode}\n')
import rdkit, torch_geometric; print('rdkit', rdkit.__version__, 'pyg', torch_geometric.__version__)

## 1. Unpack the project bundle into /tmp/molbench (only results and logs go to /kaggle/working)

In [ ]:
import traceback
def fail(msg):
    open('/kaggle/working/error.txt', 'a').write(time.strftime('%H:%M:%S ') + msg + '\n' + traceback.format_exc() + '\n')
    open('/kaggle/working/progress.txt', 'a').write(time.strftime('%H:%M:%S ') + 'ERROR ' + msg + '\n')
def find_input():
    # Kaggle mounts datasets either at /kaggle/input/<slug> or at /kaggle/input/datasets/<user>/<slug>
    cands = ['/kaggle/input/molbench-bundle'] + glob.glob('/kaggle/input/*/*/molbench-bundle') + glob.glob('/kaggle/input/*/molbench-bundle')
    for c in cands:
        if os.path.isdir(c) and (os.path.isdir(os.path.join(c, 'project')) or os.path.exists(os.path.join(c, 'project.zip'))):
            return c
    for root, dirs, files in os.walk('/kaggle/input'):
        if root.count(os.sep) > 6: continue
        if 'project' in dirs or 'project.zip' in files:
            return root
    return None
INP = None
for _ in range(60):   # the dataset version can take a few minutes to become visible after a push
    INP = find_input()
    if INP: break
    print('waiting for dataset input ...', os.listdir('/kaggle/input') if os.path.isdir('/kaggle/input') else 'no /kaggle/input', flush=True)
    time.sleep(10)
try:
    print('input dir:', INP, os.listdir(INP))
except Exception:
    fail('dataset input missing'); raise
WORK = '/tmp/molbench'
shutil.rmtree(WORK, ignore_errors=True)
if os.path.isdir(os.path.join(INP, 'project')):
    shutil.copytree(os.path.join(INP, 'project'), WORK)
else:
    zipfile.ZipFile(os.path.join(INP, 'project.zip')).extractall(WORK)
os.makedirs(os.path.join(WORK, 'data/pretrained'), exist_ok=True)
w = glob.glob(os.path.join(INP, '**', 'mat_pretrained_weights.pt'), recursive=True)
if w:
    shutil.copy(w[0], os.path.join(WORK, 'data/pretrained/mat_pretrained_weights.pt'))
else:
    zipfile.ZipFile(os.path.join(INP, 'weights.zip')).extractall(os.path.join(WORK, 'data/pretrained'))
os.chdir(WORK)
print(sorted(os.listdir(WORK)))
print('weights MB', os.path.getsize('data/pretrained/mat_pretrained_weights.pt') // 1_000_000)
print('feature cache MB', os.path.getsize('data/processed/features/mol_cache.pkl') // 1_000_000)
print('tuning present:', os.path.exists('results/tuning/best_configs.json'))
open('/kaggle/working/progress.txt', 'a').write(time.strftime('%H:%M:%S ') + 'bundle unpacked\n')

In [ ]:
SRC = os.path.join(WORK, 'src')
sys.path.insert(0, SRC)
os.environ['PYTHONPATH'] = SRC + os.pathsep + os.environ.get('PYTHONPATH', '')
import molbench; print('molbench', molbench.__version__, 'from', molbench.__file__)
def progress(msg):
    with open('/kaggle/working/progress.txt', 'a') as fh:
        fh.write(time.strftime('%H:%M:%S ') + msg + '\n')
    print(msg, flush=True)
progress('package importable')

## 2. Fidelity check (re-implementation equals the authors' code; checkpoint loads completely)

In [ ]:
r = subprocess.run([sys.executable, '-m', 'pytest', 'tests/test_mat_fidelity.py', '-q', '-p', 'no:cacheprovider'],
                   capture_output=True, text=True, env=dict(os.environ))
print(r.stdout[-1500:]); print(r.stderr[-800:])
progress(f'fidelity tests exit code {r.returncode}')
if r.returncode != 0:
    open('/kaggle/working/error.txt', 'w').write(r.stdout[-4000:] + r.stderr[-4000:])
assert r.returncode == 0, 'fidelity tests failed'

## 2b. Complete the hyperparameter search for these tasks on GPU 0 (resume-safe: rows already in results/tuning/tuning_runs.csv from the local run are reused; only missing configurations are trained)

In [ ]:
env = dict(os.environ, CUDA_VISIBLE_DEVICES='0', MOLBENCH_DEVICE='cuda', PYTHONUNBUFFERED='1')
cmd = [sys.executable, '-u', 'scripts/02_tune.py', '--tasks'] + ['estrogen-alpha'] + ['--workers', '4', '--threads', '1']
t0 = time.time()
r = subprocess.run(cmd, env=env, capture_output=True, text=True)
print(r.stdout[-3000:]); print(r.stderr[-1500:])
progress(f'tuning exit code {r.returncode} after {(time.time()-t0)/60:.1f} min')
shutil.copy('results/tuning/best_configs.json', '/kaggle/working/best_configs.json')
shutil.copy('results/tuning/tuning_runs.csv', '/kaggle/working/tuning_runs.csv')
assert r.returncode == 0, 'tuning failed'

## 3. Run `03_run_tier1.py` (GPU0 = random split, GPU1 = scaffold split; 2 worker(s) per process)

In [ ]:
os.makedirs('results/raw', exist_ok=True)
procs = {}
if N_GPU >= 1:      # one process per GPU (GPU index wraps if only one GPU is present)
    plan = [(0 % N_GPU, 'random', 'cuda', 2), (1 % N_GPU, 'scaffold', 'cuda', 2)]
else:               # CPU fallback (account without GPU): both splits, workers = CPU count
    plan = [(0, 'random', 'cpu', max(1, os.cpu_count() // 2)), (0, 'scaffold', 'cpu', max(1, os.cpu_count() // 2))]
progress(f'execution plan: {plan}')
for gpu, split, dev, nw in plan:
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu) if dev == 'cuda' else '', MOLBENCH_DEVICE=dev, PYTHONUNBUFFERED='1')
    log = open(f'results/tier1_{split}.log', 'w')
    procs[split] = subprocess.Popen([sys.executable, '-u', 'scripts/03_run_tier1.py', '--splits', split,
                                     '--workers', str(nw), '--threads', '1', '--out', f'results/raw/tier1_{split}.csv'] + ['--tasks', 'estrogen-alpha'],
                                    env=env, stdout=log, stderr=subprocess.STDOUT)
t0 = time.time()
def n_rows(f):
    return (sum(1 for _ in open(f)) - 1) if os.path.exists(f) else 0
while any(p.poll() is None for p in procs.values()):
    time.sleep(120)
    done = {s: n_rows(f'results/raw/tier1_{s}.csv') for s in procs}
    for s_ in procs:
        if os.path.exists(f'results/tier1_{s_}.log'):
            shutil.copy(f'results/tier1_{s_}.log', f'/kaggle/working/tier1_{s_}.log')
        if os.path.exists(f'results/raw/tier1_{s_}.csv'):
            shutil.copy(f'results/raw/tier1_{s_}.csv', f'/kaggle/working/tier1_{s_}.csv')
    progress(f'{(time.time()-t0)/60:5.1f} min  completed runs: {done}')
for s, p in procs.items():
    print(s, 'exit code', p.returncode)
    print(open(f'results/tier1_{s}.log').read()[-1500:])
    progress(f'{s} process exit code {p.returncode}')

## 4. Merge and summarise

In [ ]:
import pandas as pd
parts = [pd.read_csv(f) for f in ['results/raw/tier1_random.csv', 'results/raw/tier1_scaffold.csv'] if os.path.exists(f)]
df = pd.concat(parts, ignore_index=True)
df.to_csv('/kaggle/working/tier1_runs.csv', index=False)
for s_ in ['random', 'scaffold']:
    if os.path.exists(f'results/tier1_{s_}.log'):
        shutil.copy(f'results/tier1_{s_}.log', f'/kaggle/working/tier1_{s_}.log')
print(len(df), 'rows;', int((df['error'].fillna('') != '').sum()), 'errors')
print(df.groupby(['model', 'split'])[['roc_auc', 'rmse', 'epochs_run', 'wall_time_s']].mean().round(3))
print(df[df['error'].fillna('') != ''][['model', 'task', 'split', 'seed', 'error']].head(20))
progress('merged output written')

The merged file `/kaggle/working/tier1_runs.csv` is the notebook output; copy it into the local project's `results/raw/` and continue with `scripts/05_stats.py`.